# Phase 3: Geospatial assignment and spatial joins

In this notebook we implement the geospatial assignment. We take the raw synthetic citizen telemetry that we created with the open geographic data (OGD) layers from the City of Vienna (including district administrative boundaries, pedestrian zones, and cycle path networks).

The spatial queries and nearest-neighbour distance matching bind each telemetry record to its respective path and infrastructure context. This allows us to export the aggregated throughput counts and static choropleth maps.

*Note*: As the Heterogeneity Human Activity Recognition (HHAR) dataset lacks GPS telemetry, coordinates are generated programmatically using a constant-velocity local flat-earth approximation. Trajectories represent linear paths rather than actual road-network routing.


In [19]:
import os
import sys
import logging
from pathlib import Path
from pyspark.sql import SparkSession
from src.step_08_bootstrapping import setup_winutils
# Climb up from the notebook's folder to find the true project workspace root
notebook_dir = Path(os.getcwd())
PROJECT_ROOT = notebook_dir.parent if (notebook_dir.parent / "src").exists() else notebook_dir
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
# Configure Windows-specific local Spark settings dynamically
if os.name == 'nt':
    setup_winutils(PROJECT_ROOT)
    os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"
# Check if an active Spark session already exists or is configured in the environment.
active_session = SparkSession.getActiveSession()
if active_session is not None:
    spark = active_session
    logging.info("Reusing active Spark Session.")
else:
    logging.info("Spawning adaptive geospatial Spark Session environment...")
    spark_builder = (
        SparkSession.builder
        .appName('Vienna-District-Assignment')
        .config('spark.sql.shuffle.partitions', '4')
    )
    if os.name == 'nt':
        spark_builder = (
            spark_builder
            .config('spark.driver.host', '127.0.0.1')
            .config('spark.pyspark.python', sys.executable)
            .config('spark.pyspark.driver.python', sys.executable)
        )
    master_url = os.environ.get("SPARK_MASTER")
    if not master_url and not any(env.startswith("SPARK_") for env in os.environ):
        spark_builder = spark_builder.master("local[*]") \
                                     .config("spark.driver.memory", "4g")
    spark = spark_builder.getOrCreate()
logging.info(f'Project root: {PROJECT_ROOT}')


2026-07-08 14:52:05,242 - INFO - Hadoop environment path configuration active: HADOOP_HOME=c:\Users\fedka\Documents\GitHub\Geospatial Repletion & Saturation Modelling\data\winutils
2026-07-08 14:52:05,255 - INFO - Spawning adaptive geospatial Spark Session environment...
2026-07-08 14:52:07,152 - INFO - Project root: c:\Users\fedka\Documents\GitHub\Geospatial Repletion & Saturation Modelling


## 3.1 Input telemetry ingestion

The ingestion stage loads the synthesized citizen telemetry from the compressed, columnar Parquet database:
*   **Path**: `data/processed/synthetic_telemetry.parquet`
*   **Fallback**: If the master synthetic telemetry is missing, we create the minimal telemetry generator (`create_minimal_telemetry`) and produce a local diagnostic dataset containing 30 synthetic agents.


In [20]:
from src._create_minimal_telemetry import create_minimal_telemetry
telemetry_path = PROJECT_ROOT / "data" / "processed" / "synthetic_telemetry.parquet"
if not telemetry_path.exists():
    create_minimal_telemetry(PROJECT_ROOT, num_agents=30, samples_per_agent=500)
    spark.catalog.clearCache()
telemetry_df = spark.read.parquet(str(telemetry_path))
print(f"Telemetry rows: {telemetry_df.count()}")
telemetry_df.groupBy("Activity").count().show()


Telemetry rows: 7500000
+--------+-------+
|Activity|  count|
+--------+-------+
|    walk|3000000|
|    bike|2250000|
|   stand|2250000|
+--------+-------+



## 3.2 Geospatial assignment and spatial R-tree indexing

To resolve the spatial context for the stream, coordinates are mapped against the Vienna OGD layers. The algorithm utilises the R-tree (an optimised spatial indexing structure) via **Shapely STRtree** objects on the driver to execute point-in-polygon and distance-to-curve queries:

1.  **Administrative District Matching**: Telemetry coordinates are evaluated against the district boundaries using a standard point-in-polygon inclusion test:
    $$\mathbf{p} = (\lambda, \phi) \in \mathcal{P}_{\text{district}}$$
2.  **Pedestrian Zone Assignment (Walking)**: For records classified as `walk`, the pedestrian zone R-tree index (`STRtree`) is queried to identify candidate overlapping polygons, returning the specified pedestrian zone label.
3.  **Cycle Path Mapping (Biking)**: For records classified as `bike`, coordinates are matched against the bike path R-tree index. The algorithm calculates the nearest segment distance in degrees, converts it to metres, and binds the record to the cycle path if the distance is within the tolerance limit:
    $$d_{\text{metres}} = d_{\text{degrees}} \cdot M_{\text{lat}} \le 150 \text{ metres}$$
    where $M_{\text{lat}} = 111,320.0 \text{ m/degree}$.


> [!NOTE]
> **Memory efficiency justification**
> The spatial context assignment evaluates point-in-polygon inclusions and nearest segment distances using Shapely's `STRtree` spatial indexing. This is performed on the driver node for maximum CPU instruction cache efficiency. The input DataFrame is limited to the synthetic telemetry pool (1.5 million records maximum in our full bootstrap, and 15,000 records in the minimal diagnostic dataset). Collecting these rows to the driver is safe as the total memory footprint does not exceed 100 MB, which is well below the driver's memory allocation limits.


In [ ]:
from src.step_13_geospatial_districts import run_district_assignment
outputs = run_district_assignment(spark, PROJECT_ROOT)
outputs


2026-07-08 14:52:11,261 - INFO - Layer already cached: vienna_districts.geojson
2026-07-08 14:52:11,264 - INFO - Layer already cached: vienna_pedestrian_zones.geojson
2026-07-08 14:52:11,268 - INFO - Layer already cached: vienna_bike_paths.geojson
2026-07-08 14:52:13,912 - INFO - Loading cached street graph: vienna_walk_network.graphml
2026-07-08 14:54:04,760 - INFO - Collecting telemetry data to driver for spatial context assignment...


In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, LongType
summary_schema = StructType([
    StructField("district_name", StringType(), True),
    StructField("district_number", StringType(), True),
    StructField("Activity", StringType(), True),
    StructField("time_window_index", LongType(), True),
    StructField("count", LongType(), True)
])
summary_df = spark.read.schema(summary_schema).csv(
    f"{PROJECT_ROOT}/data/geospatial_output/district_activity_counts.csv",
    header=True
)
print(f"District summary rows: {summary_df.count()}")
print(f"Unique districts: {summary_df.select('district_name').distinct().count()}")
from pyspark.sql.functions import sum as spark_sum, col
district_totals = (
    summary_df.groupBy("district_name", "district_number")
    .agg(spark_sum("count").alias("simulated_record_count"))
    .orderBy(col("simulated_record_count").desc())
)
print("\nTop districts by simulated record count:")
district_totals.show(12)


District summary rows: 101
Unique districts: 24

Top districts by simulated record count:
+-------------+---------------+----------------------+
|district_name|district_number|simulated_record_count|
+-------------+---------------+----------------------+
| Innere Stadt|             01|                295600|
|    Favoriten|             10|                170000|
| Leopoldstadt|             02|                130000|
|   Donaustadt|             22|                130000|
|    Ottakring|             16|                 67500|
|    Mariahilf|             06|                 60600|
|   Alsergrund|             09|                 55100|
|    Simmering|             11|                 50000|
|     Meidling|             12|                 50000|
|  Floridsdorf|             21|                 50000|
|      Liesing|             23|                 49800|
|      Währing|             18|                 43400|
+-------------+---------------+----------------------+
only showing top 12 rows



## 3.3 Data sources and spatial provenance

*   **Sensor Telemetry Source**: UCI Heterogeneity Human Activity Recognition (HHAR) Dataset (licensed under CC BY 4.0).
*   **Spatial Reference Layers**: City of Vienna Open Government Data (OGD) (licensed under CC BY 4.0 AT), including Bezirksgrenzen (districts), Fußgängerzonen (pedestrian zones), and Radwege (bike paths).
*   **Pipeline Scope**: All coordinate data points are synthetic. The output counts and visualisations serve as a proof-of-concept validation of the Spark and spatial indexing pipeline; they do not represent actual mobility, foot traffic, or transit congestion levels in Vienna.


In [ ]:
try:
    logging.info("Shutting down Spark Session...")
finally:
    spark.stop()
    logging.info("Spark Session terminated successfully.")


## 3.4 Interactive real-time telemetry playback app

The following interactive Leaflet visualisation displays our synthetic citizens moving along their street routes in real-time. 

*   **Controls**: Use the play/pause button, the scrubbing slider, or the speed control bar at the bottom left to play back the simulation.
*   **Legend**: Walking citizens are styled in **green**, biking citizens in **blue**, and stationary citizens in **purple**.


In [ ]:
from IPython.display import IFrame
IFrame(src="../data/geospatial_output/interactive_citizens_map.html", width="100%", height=600)


**Figure 12**: Interactive real-time Leaflet simulation playback map animating synthetic citizens along Vienna's transit corridors and walking zones.
